In [0]:
import json

# STEP 1 : GET PARAMETERS FROM ADF
dbutils.widgets.text("folder_path",      "")
dbutils.widgets.text("expected_columns", "")
dbutils.widgets.text("file_format",      "")
dbutils.widgets.text("file_name",      "")

file_name        = dbutils.widgets.get("file_name")
folder_path      = dbutils.widgets.get("folder_path")
file_format      = dbutils.widgets.get("file_format").lower()
expected_columns = dbutils.widgets.get("expected_columns").split(",") 
 # ensure list
expected_set     = set(expected_columns)

print(f"File Format      : {file_format}")
print(f"Expected Columns : {sorted(expected_set)}")

# STEP 2 : READ ONLY 1 ROW TO GET HEADERS
if file_format == "json":
    df = spark.read.option("multiline", "true").format("json").load(folder_path).limit(1)
elif file_format == "csv":
    df = spark.read.option("header", "true").format("csv").load(folder_path).limit(1)

actual_columns = [c for c in df.columns if not c.startswith("_")]
actual_set     = set(actual_columns)

print(f"Actual Columns   : {sorted(actual_set)}")

# STEP 3 : CHECK 1 - COUNT MATCH
expected_count = len(expected_columns)
actual_count   = len(actual_columns)
count_match    = expected_count == actual_count

print(f"\n── Column Count Check ──")
print(f"Expected Count : {expected_count}")
print(f"Actual Count   : {actual_count}")
print(f"Count Match    : {'YES' if count_match else ' NO'}")

# STEP 4 : CHECK 2 - MISSING / EXTRA COLUMNS
missing_columns = sorted(list(expected_set - actual_set))  # in config but not in file
extra_columns   = sorted(list(actual_set - expected_set))  # in file but not in config

print(f"\n── Column Name Check ──")
print(f"Missing Columns : {missing_columns}")   #  must fix
print(f"Extra Columns   : {extra_columns}")     # allowed

# STEP 5 : CHECK 3 - EXACT NAME MATCH (case sensitive)
case_mismatch = []
for exp_col in expected_columns:
    for act_col in actual_columns:
        if exp_col.lower() == act_col.lower() and exp_col != act_col:
            case_mismatch.append({
                "expected": exp_col,
                "actual"  : act_col,
                "issue"   : "Case mismatch"
            })

print(f"\n── Case Sensitivity Check ──")
if case_mismatch:
    for c in case_mismatch:
        print(f"Expected: '{c['expected']}' | Got: '{c['actual']}' | {c['issue']}")
else:
    print(f" All column names match exactly (case sensitive)")

# STEP 6 : CHECK 4 - DUPLICATE COLUMNS
seen = set()
duplicates = []
for col in actual_columns:
    if col in seen:
        duplicates.append(col)
    else:
        seen.add(col)

print(f"\n── Duplicate Column Check ──")
if duplicates:
    print(f"Duplicate Columns Found: {duplicates}")
else:
    print(" No duplicate columns")

# STEP 7 : FINAL RESULT
all_checks_passed = (
    len(missing_columns) == 0 and
    len(case_mismatch)   == 0 and
    len(duplicates)      == 0
)

print(f"\n{'='*50}")
print(f"FINAL RESULT : {'PASSED' if all_checks_passed else 'FAILED'}")
print(f"{'='*50}")

if not all_checks_passed:
    result = {
        "status"        : "FAILED",
        "file_name"     : file_name,
        "expected_count": expected_count,
        "actual_count"  : actual_count,
        "missing"       : missing_columns,
        "extra"         : extra_columns,
        "case_mismatch" : case_mismatch,
        "duplicates"    : duplicates
    }
else:
    result = {
        "status"        : "PASSED",
        "file_name"     : file_name,
        "expected_count": expected_count,
        "actual_count"  : actual_count,
        "missing"       : [],
        "extra"         : extra_columns,
        "case_mismatch" : [],
        "duplicates"    : []
    }

print(json.dumps(result, indent=2))
dbutils.notebook.exit(json.dumps(result))
